# Análisis Arsenal — Jornada 26 (Brentford 1-1 Arsenal)
## ETL paso a paso con Python

Este cuaderno lee los **3 partidos** en formato Opta (JSON) y genera una tabla limpia (CSV) para Power BI.

| Bloque | Partido | Resultado |
|---|---|---|
| Postpartido | Brentford vs Arsenal (J26) | 1-1 |
| Prepartido Arsenal | Arsenal vs Sunderland (J25) | 3-0 |
| Prepartido rival | Newcastle vs Brentford (J25) | 2-3 |

**Cómo usar este cuaderno:** ejecuta cada celda en orden con `Shift + Enter`. Lee el texto antes de cada bloque de código.


## Paso 1 — Importar librerías

`json` viene con Python y sirve para leer los archivos. `pandas` es la librería de tablas (como Excel dentro de Python).
`pathlib` nos ayuda a manejar rutas de carpetas de forma ordenada.


In [1]:
import json
from pathlib import Path
import pandas as pd

print('Librerías cargadas correctamente')


Librerías cargadas correctamente


## Paso 2 — Definir dónde están los archivos

Indicamos la carpeta `data/raw` (donde están los 3 JSON) y `data/processed` (donde guardaremos el CSV).
El diccionario `FICHEROS` asocia cada **bloque** del análisis con su archivo. Esto hace el código reutilizable:
si mañana analizas otra jornada, solo cambias estas rutas.


In [2]:
# Carpeta del proyecto (la carpeta donde está este cuaderno)
BASE = Path.cwd()
RAW = BASE / 'data' / 'raw'
OUT = BASE / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)  # crea la carpeta de salida si no existe

FICHEROS = {
    'postpartido':        RAW / 'postpartido_J26_Brentford_Arsenal.json',
    'prepartido_arsenal': RAW / 'prepartido_arsenal_J25_Arsenal_Sunderland.json',
    'prepartido_rival':   RAW / 'prepartido_brentford_J25_Newcastle_Brentford.json',
}

# Comprobamos que los 3 archivos existen antes de seguir
for bloque, ruta in FICHEROS.items():
    print(f'{bloque:20s} -> {"OK" if ruta.exists() else "NO ENCONTRADO"}  ({ruta.name})')


postpartido          -> OK  (postpartido_J26_Brentford_Arsenal.json)
prepartido_arsenal   -> OK  (prepartido_arsenal_J25_Arsenal_Sunderland.json)
prepartido_rival     -> OK  (prepartido_brentford_J25_Newcastle_Brentford.json)


## Paso 3 — Abrir un JSON y entender su estructura (EXTRACT)

Abrimos el partido del Arsenal y miramos qué hay dentro. Cada JSON tiene dos grandes claves:
- **`matchInfo`**: datos del partido (equipos, fecha, jornada, estadio).
- **`liveData`**: el marcador (`matchDetails`) y la lista de **eventos** (`event`): cada pase, tiro, falta...


In [3]:
with open(FICHEROS['postpartido'], 'r', encoding='utf-8') as f:
    partido = json.load(f)

print('Claves principales:', list(partido.keys()))
print('Descripción      :', partido['matchInfo']['description'])
print('Jornada          :', partido['matchInfo']['week'])
print('Marcador final   :', partido['liveData']['matchDetails']['scores']['ft'])
print('Nº de eventos    :', len(partido['liveData']['event']))


Claves principales: ['matchInfo', 'liveData']
Descripción      : Brentford vs Arsenal
Jornada          : 26
Marcador final   : {'home': 1, 'away': 1}
Nº de eventos    : 1562


Veamos **un evento** de ejemplo. Lo importante es el campo `typeId` (qué tipo de acción es)
y `outcome` (si salió bien =1 o mal =0). `contestantId` dice de qué equipo es.


In [4]:
ejemplo = partido['liveData']['event'][100]
print('typeId      :', ejemplo['typeId'], '(tipo de evento)')
print('contestantId:', ejemplo['contestantId'], '(equipo)')
print('playerName  :', ejemplo.get('playerName'))
print('outcome     :', ejemplo.get('outcome'), '(1=éxito, 0=fallo)')


typeId      : 49 (tipo de evento)
contestantId: 4dsgumo7d4zupm2ugsvm4zm4d (equipo)
playerName  : Cristhian Mosquera
outcome     : 1 (1=éxito, 0=fallo)


## Paso 4 — Funciones auxiliares

Creamos dos funciones pequeñas para no repetir código:
- `cargar(ruta)`: abre un JSON y lo devuelve.
- `quals(evento)`: devuelve los *qualifiers* (datos extra) de un evento como diccionario. 
  Los usamos, por ejemplo, para saber si una tarjeta es amarilla o roja.


In [5]:
def cargar(ruta):
    with open(ruta, 'r', encoding='utf-8') as f:
        return json.load(f)

def quals(evento):
    return {q['qualifierId']: q.get('value') for q in evento.get('qualifier', [])}

print('Funciones definidas')


Funciones definidas


## Paso 5 — Diccionario de tipos de evento Opta

Cada acción tiene un `typeId`. Estos son los que usamos para las métricas:

| typeId | Significado |
|---|---|
| 1 | Pase |
| 3 | Regate (take-on) |
| 4 | Falta |
| 6 | Córner concedido/ganado |
| 7 | Despeje |
| 8 | Intercepción |
| 12 | Entrada (tackle) |
| 13 | Tiro fuera |
| 14 | Tiro al palo |
| 15 | Tiro parado (a puerta) |
| 16 | GOL |
| 17 | Tarjeta |
| 44 | Duelo aéreo |

**Tiros a puerta** = 15 + 16. **Tiros totales** = 13 + 14 + 15 + 16.


In [6]:
TIPOS_TIRO = {13, 14, 15, 16}   # cualquier tiro
TIROS_PUERTA = {15, 16}         # tiro a puerta (parado o gol)
print('Constantes listas')


Constantes listas


## Paso 6 — Calcular métricas de un equipo (TRANSFORM)

Aquí está el corazón de la ETL. La idea es siempre la misma: **filtrar los eventos de un equipo
por su `typeId` y contarlos**.

**Dos detalles importantes** que distinguen un análisis riguroso:
1. Las **faltas** se registran dos veces en Opta: quien la comete (`outcome=0`) y quien la recibe (`outcome=1`).
   Contamos solo `outcome=0` para 'faltas cometidas'.
2. Los **córners** también se registran doble: a favor (`outcome=1`) y en contra (`outcome=0`).
   Contamos solo `outcome=1`.


In [7]:
def metricas_equipo(eventos, team_id):
    ev = [e for e in eventos if e.get('contestantId') == team_id]

    pases = [e for e in ev if e['typeId'] == 1]
    pases_ok = [e for e in pases if e.get('outcome') == 1]
    tiros = [e for e in ev if e['typeId'] in TIPOS_TIRO]
    tiros_p = [e for e in ev if e['typeId'] in TIROS_PUERTA]
    aereos = [e for e in ev if e['typeId'] == 44]
    regates = [e for e in ev if e['typeId'] == 3]

    tarjetas = [e for e in ev if e['typeId'] == 17]
    amarillas = sum(1 for t in tarjetas if 31 in quals(t))
    rojas = sum(1 for t in tarjetas if 32 in quals(t) or 33 in quals(t))

    return {
        'pases': len(pases),
        'pases_completados': len(pases_ok),
        'precision_pase_pct': round(100*len(pases_ok)/len(pases), 1) if pases else 0,
        'tiros_totales': len(tiros),
        'tiros_a_puerta': len(tiros_p),
        'goles': sum(1 for e in ev if e['typeId'] == 16),
        'entradas': sum(1 for e in ev if e['typeId'] == 12),
        'intercepciones': sum(1 for e in ev if e['typeId'] == 8),
        'despejes': sum(1 for e in ev if e['typeId'] == 7),
        'faltas_cometidas': sum(1 for e in ev if e['typeId'] == 4 and e.get('outcome') == 0),
        'corners': sum(1 for e in ev if e['typeId'] == 6 and e.get('outcome') == 1),
        'duelos_aereos': len(aereos),
        'duelos_aereos_ganados': sum(1 for e in aereos if e.get('outcome') == 1),
        'regates': len(regates),
        'regates_exitosos': sum(1 for e in regates if e.get('outcome') == 1),
        'amarillas': amarillas,
        'rojas': rojas,
    }

print('Función metricas_equipo lista')


Función metricas_equipo lista


## Paso 7 — Procesar un partido entero

Esta función toma un partido, identifica los dos equipos (local y visitante), calcula sus métricas
y añade la **posesión** (estimada como el % de pases del equipo sobre el total del partido).


In [8]:
def procesar(bloque, ruta):
    d = cargar(ruta)
    mi = d['matchInfo']
    ft = d['liveData']['matchDetails']['scores']['ft']
    eventos = d['liveData']['event']

    equipos = {c['position']: c for c in mi['contestant']}
    filas = []
    for lado in ['home', 'away']:
        equipo = equipos[lado]
        m = metricas_equipo(eventos, equipo['id'])
        filas.append({
            'bloque': bloque,
            'jornada': mi['week'],
            'fecha': mi['date'][:10],
            'partido': mi['description'],
            'equipo': equipo['name'],
            'lado': 'local' if lado == 'home' else 'visitante',
            'goles_marcador': ft[lado],
            **m,
        })

    # Posesión = cuota de pases sobre el total del partido
    total_pases = sum(f['pases'] for f in filas)
    for f in filas:
        f['posesion_pct'] = round(100*f['pases']/total_pases, 1) if total_pases else 0
    return filas

# Probamos con el partido foco
prueba = procesar('postpartido', FICHEROS['postpartido'])
for fila in prueba:
    print(fila['equipo'], '| posesión', fila['posesion_pct'], '% | tiros', fila['tiros_totales'])


Brentford | posesión 41.5 % | tiros 12
Arsenal | posesión 58.5 % | tiros 7


## Paso 8 — Procesar los 3 partidos y montar la tabla

Recorremos los tres bloques, juntamos todas las filas y las metemos en un `DataFrame` de pandas
(la tabla). Ordenamos las columnas para que se lean bien.


In [9]:
todas = []
for bloque, ruta in FICHEROS.items():
    print('Procesando', bloque)
    todas.extend(procesar(bloque, ruta))

df = pd.DataFrame(todas)

cols = ['bloque','jornada','fecha','partido','equipo','lado','goles_marcador',
        'posesion_pct','tiros_totales','tiros_a_puerta','pases','pases_completados',
        'precision_pase_pct','regates','regates_exitosos','duelos_aereos',
        'duelos_aereos_ganados','entradas','intercepciones','despejes',
        'faltas_cometidas','corners','amarillas','rojas']
df = df[cols]
df


Procesando postpartido
Procesando prepartido_arsenal
Procesando prepartido_rival


,bloque,jornada,fecha,partido,equipo,lado,goles_marcador,posesion_pct,tiros_totales,tiros_a_puerta,...,regates_exitosos,duelos_aereos,duelos_aereos_ganados,entradas,intercepciones,despejes,faltas_cometidas,corners,amarillas,rojas
0,postpartido,26,2026-02-12,Brentford vs Arsenal,Brentford,local,1,41.5,12,8,...,6,28,7,29,12,11,12,6,4,0
1,postpartido,26,2026-02-12,Brentford vs Arsenal,Arsenal,visitante,1,58.5,7,5,...,3,28,21,37,4,22,11,4,2,0
2,prepartido_arsenal,25,2026-02-07,Arsenal vs Sunderland,Arsenal,local,3,49.1,16,10,...,6,33,17,23,12,15,11,5,1,0
3,prepartido_arsenal,25,2026-02-07,Arsenal vs Sunderland,Sunderland,visitante,0,50.9,5,4,...,5,33,16,32,6,13,8,2,2,0
4,prepartido_rival,25,2026-02-07,Newcastle United vs Brentford,Newcastle United,local,2,53.2,16,8,...,10,38,22,33,4,9,14,9,2,0
5,prepartido_rival,25,2026-02-07,Newcastle United vs Brentford,Brentford,visitante,3,46.8,11,9,...,6,38,16,21,3,15,18,3,2,0


## Paso 9 — Exportar a CSV (LOAD)

Guardamos la tabla en `data/processed/metricas_equipo.csv`. Usamos `encoding='utf-8-sig'`
para que Excel y Power BI muestren bien los acentos. ¡Este es el archivo que cargaremos en Power BI!


In [10]:
salida = OUT / 'metricas_equipo.csv'
df.to_csv(salida, index=False, encoding='utf-8-sig')
print('Guardado en:', salida)
print('Filas:', len(df), '| Columnas:', len(df.columns))


Guardado en: C:\Users\ivan1\OneDrive\Escritorio\proyecto_arsenal\data\processed\metricas_equipo.csv
Filas: 6 | Columnas: 24


## ¡Listo!

Ya tienes `metricas_equipo.csv` generado. En el siguiente paso lo cargaremos en **Power BI**
para construir el informe visual de los tres bloques.
